# Compare inference batches with a conditional drift step

Run ordinary inference once without a reference. Then run another batch with the
first execution's ARN as `ReferenceExecutionArn`. Postprocess runs customer prediction
analysis. The HasReference gate then starts DriftComparison only when a reference
is supplied. It writes Evidently reports to S3 and the same inference MLflow run.
The reference is read from saved outputs; it is not scored again.

Use notebook 02 to publish Processing inference with `monitoring.enabled=True`.
Its approved image must contain Evidently. MLflow is optional; reports still go to S3.
The first demonstration should keep the same approved model version for both runs.

## Select the published pipeline

- Load the publication saved by notebook 02 and use its exact pipeline version.
- Select the model and scoring CSV; keep the reference empty for the first run.
- Use the notebook's execution-role credentials in the publication's AWS Region.


In [ ]:
from pathlib import Path
import json, sys
import boto3

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-models/src").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models/src"), str(ROOT / "mlops-models")]
from mlops_models.pipelines.inspect import wait_for_pipeline, collect_execution
from mlops_models.pipelines.tracking import sync_execution

namespace = json.loads((ROOT / "notebooks/config.json").read_text())["namespace"]
publication = json.loads((ROOT / ".runs/source-deploy" / namespace / "publication.json").read_text())
assert publication.get("monitoring", {}).get("enabled"), "Enable monitoring and republish using notebook 02."
session = boto3.Session(region_name=publication["region"])
sm, s3 = session.client("sagemaker"), session.client("s3")
pipeline = publication["pipelines"]["inference"]
model_id = "usecase2"
model_package_arn = ""  # Pin the approved Development version for both comparison runs.
current_data_uri = ""  # Supply the scoring CSV's s3:// URI.
reference_execution_arn = ""  # First run: empty. Next run: paste the first successful execution ARN.
execution_arn = ""  # Or paste an existing inference ARN to inspect without starting a job.
tracking = publication.get("tracking")
WORK = ROOT / ".runs/inference-monitoring" / namespace
WORK.mkdir(parents=True, exist_ok=True)


## Start an inference execution

- Set `start_execution=True` to start the selected batch; otherwise this cell submits nothing.
- Pass the optional reference ARN to enable DriftComparison; omit it to skip that job.
- Keep the returned ARN: use it as the reference when running the next batch.

For the second run, change `current_data_uri` and set `reference_execution_arn` to
the first run's ARN. Keeping the same CSV is also useful: it checks that upstream
compute can be cached while DriftComparison runs the newly requested comparison.
Select a fixed reference explicitly; the notebook never chooses the latest run.


In [ ]:
start_execution = False
if start_execution:
    assert current_data_uri.startswith("s3://") and current_data_uri.endswith(".csv"), "Supply a scoring CSV URI."
    assert model_id in publication["models"], "Choose a published ModelId."
    parameters = {"ModelId": model_id, "CsvInputUri": current_data_uri}
    if model_package_arn:
        parameters["ModelPackageArn"] = model_package_arn
    if reference_execution_arn:
        parameters["ReferenceExecutionArn"] = reference_execution_arn
    if publication["batch_runtime"] == "both":
        parameters["InferenceRuntime"] = "processing"
    response = sm.start_pipeline_execution(
        PipelineName=pipeline["name"], PipelineVersionId=pipeline["version"],
        PipelineExecutionDisplayName="inference-comparison" if reference_execution_arn else "inference-reference",
        PipelineParameters=[{"Name": name, "Value": value} for name, value in parameters.items()])
    execution_arn = response["PipelineExecutionArn"]
    (WORK / (execution_arn.rsplit("/", 1)[-1] + ".json")).write_text(json.dumps({
        "execution_arn": execution_arn, "parameters": parameters}, indent=2) + "\n")
    print("Started:", execution_arn)
else:
    print("No execution submitted. Set execution_arn above to inspect an existing run.")


## Inspect the completed inference run

- Wait for the selected execution and inspect its actual producing jobs and S3 outputs.
- Finalize that execution's MLflow status when tracking is configured, including cached-output links.
- Print comparison reports when the gate ran; no comparison job or drift reports exist without a reference.

**Where to look:** SageMaker Pipelines → this inference execution → HasReference → DriftComparison.
In the MLflow App, open the corresponding inference run. Reports are under
`outputs/DriftComparison/monitoring/`; scalar metrics start with `monitoring.` and
reference identities are recorded as tags. S3 keeps the reports even with tracking disabled.


In [ ]:
report = None
if execution_arn:
    status = wait_for_pipeline(session, execution_arn)
    print("Pipeline status:", status["PipelineExecutionStatus"])
    if tracking:
        tracked = sync_execution(session, execution_arn, {
            **tracking, "kind": "inference", "environment": publication["environment"],
            "s3_root": publication["s3_root"],
            "kms_key": publication.get("infrastructure", {}).get("output_kms_key_arn", "")})
        print("MLflow run:", tracked["run_id"], "| status:", tracked["status"])
    report = collect_execution(session, execution_arn)
    for step in report["steps"]:
        print(step["name"], step["status"], "| cached:", step["cached"],
              "| producer:", step["producer_execution_arn"])
    for name, value in report["reports"].items():
        if "/monitoring/" in name:
            print(name, json.dumps(value, indent=2))
    for artifact in report["artifacts"]:
        if artifact["relative_path"].startswith("monitoring/") and artifact["relative_path"].endswith(".html"):
            print("Report:", artifact["uri"])
else:
    print("Select an execution ARN above.")


## Read the results

| Report | Without a reference | With a reference |
| --- | --- | --- |
| Data quality | Comparison job skipped | Both batches |
| Feature drift | Comparison job skipped | Selected input-feature distributions |
| Prediction drift | Comparison job skipped | Score-distribution comparison |
| Model quality | Comparison job skipped; customer analysis still runs | Reference/current metrics and differences when both have labels |

Feature or prediction drift is a reason to investigate, not proof of degraded
accuracy. Quality metrics require actual outcomes. If those arrive later, join them
to the saved predictions by record key and call `write_report` on those aligned
frames; scoring does not need to run again. Automated outcome ingestion belongs
to the platform integration.

The default PSI threshold (0.2) and drifted-feature share (0.5) are examples to
review with the model owner. The summary records both model versions: comparing
different model versions changes the interpretation of prediction/quality differences.
An invalid supplied reference fails visibly instead of silently skipping comparison.
